# Tarea: Geometría de Datos en Producción
## Escalamiento y Discretización (Google Play Store)

**Objetivo:**
En la clase anterior demostramos cómo las variables con escalas gigantescas secuestran el espacio vectorial y destruyen los algoritmos de Machine Learning. Ahora es su turno de aplicar estos conceptos arquitectónicos al *dataset* de la Google Play Store que limpiaron la semana pasada.

**Instrucciones:**
Resuelvan los siguientes 4 retos programando directamente en las celdas de código de este Notebook. Para cada reto, respondan a la pregunta analítica utilizando un comentario o una celda Markdown.

In [4]:
import pandas as pd
import numpy as np

# Instrucción: Carga tu archivo limpio de la semana 4 ('playstore_limpio.csv').
# Si por alguna razón no lo tienes, usa el siguiente bloque para descargar y pre-limpiar lo básico:

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

# Limpieza rápida (Mockup del resultado de la Semana 4)
df.drop_duplicates(subset=['App'], inplace=True)
df = df[df['Installs'] != 'Free'] # Quitar la fila de error ("Life Made WI-Fi")
df['Installs'] = df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False).astype(float)
df['Reviews'] = df['Reviews'].astype(float)
df['Rating'] = df['Rating'].astype(float)
# Rellenar ratings nulos con la mediana para que el escalamiento no falle
df['Rating'].fillna(df['Rating'].median(), inplace=True)

print("Datos listos. Total de apps:", len(df))

Datos listos. Total de apps: 9659


C:\Users\jumbo\AppData\Local\Temp\ipykernel_27156\3140479116.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Rating'].fillna(df['Rating'].median(), inplace=True)


---
### Reto 1: La Trampa de Min-Max en un Mundo Viral
La columna `Installs` (Descargas) representa un reto enorme porque vivimos en un mundo asimétrico: hay millones de apps con 10 descargas y unas cuantas con 1,000 millones (como WhatsApp).

1. Crea una nueva columna llamada `Installs_MinMax` aplicando la fórmula $(X - Min) / (Max - Min)$.
2. Ejecuta un `.describe()` sobre tu nueva columna.
3. **Pregunta de Análisis:** Observa el percentil 75% (`75%`) de tu nueva columna comprimida. ¿Qué valor numérico tiene? Como arquitecto de datos, explica qué nos dice este número sobre el riesgo de usar *Min-Max Scaler* en variables virales (con *outliers* masivos).

In [5]:
import pandas as pd
import numpy as np

# Instrucción: Carga tu archivo limpio de la semana 4 ('playstore_limpio.csv').
# Si por alguna razón no lo tienes, usa el siguiente bloque para descargar y pre-limpiar lo básico:

url = "https://raw.githubusercontent.com/AndresSilva23/MineriaDeDatos/refs/heads/main/parserTensor/playstore_limpio.csv"
df = pd.read_csv(url)

# 1. Aplica la fórmula Min-Max a la columna 'Installs'
installs_numeric = (df['Installs'].astype(str).str.replace('+', '', regex=False).str.replace(',', '', regex=False).replace('Free', np.nan).astype(float))

df['Installs_MinMax'] = ((installs_numeric - installs_numeric.min()) / (installs_numeric.max() - installs_numeric.min()))

# 2. Muestra el resumen estadístico
print(df['Installs_MinMax'].describe())



count    10816.000000
mean         0.015499
std          0.085121
min          0.000000
25%          0.000005
50%          0.000100
75%          0.005000
max          1.000000
Name: Installs_MinMax, dtype: float64


### RESPUESTA A LA PREGUNTA (Escribe aquí tu análisis):

Observa el percentil 75% (`75%`) de tu nueva columna comprimida. ¿Qué valor numérico tiene? Como arquitecto de datos, explica qué nos dice este número sobre el riesgo de usar *Min-Max Scaler* en variables virales (con *outliers* masivos)

El percentil 75% de Installs_MinMax es 0.005. Esto significa que el 75% de las aplicaciones tiene un valor normalizado menor o igual a 0.005, mientras que una pequeña cantidad de aplicaciones concentra valores mucho más altos, esto demuestra que existe una gran diferencia entre las aplicaciones con pocas instalaciones y las aplicaciones virales. Por lo tanto, el uso de Min-Max puede ser riesgoso en variables con outliers masivos, ya que los valores extremos pueden comprimir a la mayoría de los datos cerca de 0 y hacer que las diferencias entre aplicaciones normales sean poco visibles.

---
### Reto 2: Estandarización de Calificaciones (Z-Score)
La columna `Rating` va de 1.0 a 5.0. Vamos a estandarizarla para que su media sea 0 y su desviación estándar sea 1 (Campana de Gauss). En una distribución normal, cualquier valor con un $Z-Score$ menor a $-3$ es estadísticamente una "anomalía negativa" extrema.

1. Crea la columna `Rating_Z` aplicando la fórmula $(X - \mu) / \sigma$.
2. Escribe una máscara booleana para filtrar el *DataFrame* y encontrar cuántas aplicaciones tienen un `Rating_Z` menor a `-3.0`.
3. **Pregunta de Análisis:** ¿Cuántas aplicaciones son "anomalías negativas" extremas? ¿Qué calificación original (en estrellas) tenían esas apps para merecer ese Z-Score?

In [6]:
# 1. Aplica la fórmula Z-Score a la columna 'Rating'
df['Rating_Z'] = (df['Rating'] - df['Rating'].mean()) / df['Rating'].std()

# 2. Filtra las apps con Z < -3.0
anomalias_negativas = df[ df['Rating_Z'] < -3.0 ]
print("Total de anomalías negativas:", len(anomalias_negativas))
print(anomalias_negativas[['App', 'Rating', 'Rating_Z']].head())


Total de anomalías negativas: 175
                                                App  Rating  Rating_Z
477                                      Calculator     2.6 -3.089973
518                   Just She - Top Lesbian Dating     1.9 -4.448466
520  EliteSingles – Dating for Single Professionals     2.5 -3.284043
527                          Sugar Daddy Dating App     2.5 -3.284043
549  EliteSingles – Dating for Single Professionals     2.5 -3.284043



### RESPUESTA A LA PREGUNTA:

¿Cuántas aplicaciones son "anomalías negativas" extremas? ¿Qué calificación original (en estrellas) tenían esas apps para merecer ese Z-Score?

Se encontraron 175 aplicaciones como anomalías negativas. Sus calificaciones originales eran muy bajas, con ejemplos de 1.9, 2.5 y 2.6 estrellas, lo que provocó que su Z-Score fuera menor a -3.0. Esto indica que sus calificaciones se encuentran muy alejadas por debajo del promedio del dataset.

---
### Reto 3: Categorización Comercial (El Cuchillo Lógico `pd.cut`)
El equipo de marketing no entiende de estrellas decimales (ej. 4.3). Quieren que las aplicaciones se clasifiquen en 4 etiquetas simples para el *frontend* de la tienda.

1. Utiliza `pd.cut()` sobre la columna original `Rating` para crear una nueva columna llamada `Categoria_Calidad`.
2. Utiliza los siguientes límites (`bins`): `0.0, 3.0, 4.0, 4.5, 5.0`.
3. Asigna las siguientes etiquetas (`labels`): `'Pésima', 'Regular', 'Buena', 'Excelente'`.
4. Ejecuta un `.value_counts()` para ver cuántas hay de cada una. ¿Quedaron desbalanceadas?

In [7]:
# 1 y 2. Aplica pd.cut con los bins y labels especificados
df['Categoria_Calidad'] = pd.cut(df['Rating'], bins=[0.0, 3.0, 4.0, 4.5, 5.0], labels=['Pésima', 'Regular', 'Buena', 'Excelente'])

# 4. Muestra los conteos
print(df['Categoria_Calidad'].value_counts())



Categoria_Calidad
Buena        4878
Regular      2187
Excelente    1915
Pésima        369
Name: count, dtype: int64


### RESPUESTA:

¿Quedaron desbalanceadas?

Sí, las categorías quedaron desbalanceadas. La categoría Buena tiene la mayor cantidad de aplicaciones, con 4,878 registros, mientras que Pésima tiene solamente 369. Esto significa que el modelo tendría muchas más muestras de la categoría Buena que de las demás, especialmente de Pésima.

---
### Reto 4: Cuartiles de *Engagement* (El Cuchillo Estadístico `pd.qcut`)
A diferencia del equipo de marketing, tu modelo de Machine Learning necesita grupos balanceados para no desarrollar sesgos. Vamos a clasificar las aplicaciones basándonos en su volumen de reseñas (`Reviews`).

1. Utiliza `pd.qcut()` sobre la columna `Reviews` para crear 4 grupos de igual tamaño (`q=4`). Llama a la nueva columna `Nivel_Engagement`.
2. Asigna las etiquetas: `'Bajo', 'Medio', 'Alto', 'Viral'`.
3. Ejecuta un `.value_counts()` sobre `Nivel_Engagement`.
4. **Pregunta de Análisis:** Compara este conteo con el que obtuviste en el Reto 3. ¿Por qué el algoritmo de clasificación preferiría predecir la columna `Nivel_Engagement` en lugar de la columna `Categoria_Calidad`?

In [8]:
# 1 y 2. Aplica pd.qcut para generar 4 categorías de igual tamaño
df['Nivel_Engagement'] = pd.qcut(df['Reviews'], q=4, labels=['Bajo', 'Medio', 'Alto', 'Viral'])

# 3. Muestra los conteos (Deberían ser casi idénticos en cada categoría)
print(df['Nivel_Engagement'].value_counts())



Nivel_Engagement
Bajo     2727
Viral    2704
Alto     2703
Medio    2682
Name: count, dtype: int64


#### RESPUESTA A LA PREGUNTA:

¿Por qué el algoritmo de clasificación preferiría predecir la columna `Nivel_Engagement` en lugar de la columna `Categoria_Calidad`?

El algoritmo de clasificación preferiría predecir la columna Nivel_Engagement porque sus categorías están mucho más balanceadas. Cada categoría contiene aproximadamente la misma cantidad de aplicaciones, mientras que Categoria_Calidad está desbalanceada, ya que la categoría Buena concentra una gran cantidad de registros y Pésima tiene muy pocos.